# Cafe-Focusing Tutorial

이 노트북은 `focuser.py`에 정의된 **`CafeFocuser`** 모듈을 사용하여 이미지 아웃포커싱 처리를 수행하는 튜토리얼입니다.
OpenCV와 Contour 분석을 활용해 인공지능(AI) 없이도 가볍게 객체의 경계를 찾고 배경을 흐리게 만듭니다.

### 1. 환경 설정 및 종속성 설치
필요한 라이브러리를 설치합니다. 이미 환경이 설정되어 있다면 이 단계를 건너뛰셔도 좋습니다.

In [ ]:
# 만약 라이브러리가 설치되어 있지 않다면 아래 명령을 주석 해제하여 실행하세요.
# !pip install -r requirements.txt

### 2. 필요한 라이브러리 및 CafeFocuser 로드

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from focuser import CafeFocuser

# 주피터 노트북에 이미지를 인라인으로 표시하기 위한 설정
%matplotlib inline

### 3. 입력 이미지 준비
본 예제에서는 `example_process_img/food_solo.png`를 기본 입력 이미지로 사용합니다.

In [ ]:
import os

# 이미지 경로 설정 (디렉터리 구조에 맞춰 유동적으로 변경 가능)
img_path = 'example_process_img/food_solo.png'
if not os.path.exists(img_path):
    # 만약 루트 디렉터리에 복사되어 있다면
    img_path = 'food_solo.png'

print(f"Using image path: {img_path}")
img_bgr = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 6))
plt.imshow(img_rgb)
plt.title('Original Image')
plt.axis('off')
plt.show()

### 4. CafeFocuser를 활용한 아웃포커싱 처리
두 가지 방식으로 비교해봅니다.
1. **Legacy Blend**: 기존 주피터 노트북의 AND 합성 방식 (경계선이 하드하게 끊김)
2. **Natural Alpha Blend**: 개선된 알파 채널 선형 보간 방식 (경계면이 매우 부드럽고 자연스러움)

In [ ]:
# Focuser 인스턴스 생성
focuser = CafeFocuser(
    canny_low=40,
    canny_high=150,
    mask_dilate_iter=10,
    mask_erode_iter=10,
    mask_blur_size=(21, 21),
    bg_blur_size=(13, 13)
)

# 1. 기존 (Legacy) AND 마스크 방식 실행
mixed_legacy, steps_legacy = focuser.process(img_bgr, use_alpha_blend=False)
mixed_legacy_rgb = cv2.cvtColor(mixed_legacy, cv2.COLOR_BGR2RGB)

# 2. 개선된 (Alpha) 블렌딩 방식 실행
mixed_alpha, steps_alpha = focuser.process(img_bgr, use_alpha_blend=True)
mixed_alpha_rgb = cv2.cvtColor(mixed_alpha, cv2.COLOR_BGR2RGB)

### 5. 아웃포커싱 결과 비교 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

axes[0].imshow(img_rgb)
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(mixed_legacy_rgb)
axes[1].set_title('Legacy Focusing (Bitwise AND)')
axes[1].axis('off')

axes[2].imshow(mixed_alpha_rgb)
axes[2].set_title('Improved Focusing (Alpha Blend)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

### 6. 상세 단계별 필터 시각화
CafeFocuser가 내부적으로 처리하는 주요 중간 단계들을 한눈에 시각화해봅니다.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.ravel()

# 시각화할 단계 매핑
visual_steps = [
    ('gray', '1. Grayscale', 'gray'),
    ('canny_edge', '2. Canny Edge', 'gray'),
    ('canny_dilate', '3. Dilated Edge', 'gray'),
    ('img_draw_contour', '4. Contour Detected', 'gray'),
    ('fill_mask', '5. Convex Mask', 'gray'),
    ('mask_gaussian', '6. Smoothed Mask', 'gray'),
    ('img_blur', '7. Blurred Background (Alpha)', None),
    ('mixed', '8. Final Composite (Alpha)', None)
]

for i, (key, title, cmap) in enumerate(visual_steps):
    img_step = steps_alpha[key]
    # RGB 변환 처리 (컬러 채널이 있는 경우)
    if len(img_step.shape) == 3:
        img_step = cv2.cvtColor(img_step, cv2.COLOR_BGR2RGB)
        
    axes[i].imshow(img_step, cmap=cmap)
    axes[i].set_title(title)
    axes[i].axis('off')

plt.tight_layout()
plt.show()